# GAT with SR-GNN-Style Session Readout

This notebook follows the SR-GNN session recommendation pipeline, but replaces the GGNN encoder with a Graph Attention Network. The readout stays SR-GNN-style: last-click local preference plus attention-based global preference.

## Architecture

A session prefix `[v1, v2, ..., vt]` is turned into a graph. Nodes are unique items from the prefix. Directed edges follow observed clicks, for example `v1 -> v2`; reverse edges are also built so the model can read both incoming and outgoing transition context. Repeated transitions are normalized and passed as edge weights.

The encoder is a bidirectional GAT. It uses the same item embedding table for both directions, but separate GAT weights for forward and backward transitions:

```text
session prefix
     |
directed session graph
     |
item embedding, dim=100
     |
     +--> forward GATConv: 4 heads x 25 dims, dropout=0.1, edge weights
     |
     +--> backward GATConv: 4 heads x 25 dims, dropout=0.1, edge weights
                 |
concat forward/backward states, dim=200
                 |
linear direction fusion, 200 -> 100
                 |
contextual node embeddings
                 |
SR-GNN local/global readout
                 |
scores for all items
```

GAT details used here:

- Layer type: `torch_geometric.nn.GATConv`.
- Hidden size: `100`.
- Number of GAT layers per direction: `1`.
- Attention heads: `4`.
- Per-head output size: `25`, concatenated back to `100`.
- Dropout inside GAT and after activation: `0.1`.
- Activation: `ELU`.
- Residual connection: disabled (GATConv self-loops already provide self-bias).
- Self-loops: enabled by `GATConv`.
- Edge features: unweighted adjacency. Multi-head attention learns transition importance natively.
- Direction fusion: concatenate forward and backward node states, then apply a linear layer `200 -> 100`.

In a GAT layer, a node learns attention weights over its connected items instead of treating all neighbors equally. This is why GAT is a reasonable replacement for the GGNN encoder: session graphs are small, directed, and noisy, and some transitions should matter more than others. The bidirectional setup keeps the SR-GNN idea of using both incoming and outgoing transition context.

After graph encoding, node embeddings are mapped back to the original click order. The last clicked item gives the local preference. Attention over all prefix positions gives the global preference. These two vectors are concatenated and projected into one session representation.

## Prediction

The final session vector is multiplied by the item embedding matrix. This gives one score for every candidate item. Items are ranked by score, and the top 20 are used for Precision@20 and MRR@20.

This Kaggle-ready notebook prepares raw Yoochoose and Diginetica inputs, runs the SR-GNN/TAGNN preprocessing pipeline, and trains the GAT + SR-GNN-style readout model in one file.

Expected Kaggle input directories:
- `/kaggle/input/datasets/chadgostopp/recsys-challenge-2015` containing `yoochoose-clicks.dat`
- `/kaggle/input/datasets/profalbusdumbledore/diginetica-dataset` containing `train-item-views.csv`

`Yoochoose 1/4` is skipped because it is too large for the target memory budget.

## Prepare Datasets

The Kaggle inputs contain the original raw files. This section reads the raw Kaggle files directly and prepares the in-memory prefix-label examples used by the following sections.

In [ ]:
%pip install torch_geometric

In [ ]:
import os
import random
import time
from collections import Counter
from datetime import date
from pathlib import Path

MPLCONFIGDIR = Path("/tmp/matplotlib")
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import kagglehub
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GATConv
from torch_geometric.utils import softmax as pyg_softmax

In [ ]:
KAGGLE_INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")

YOOCHOOSE_INPUT_DIR = KAGGLE_INPUT_DIR / "datasets/chadgostopp/recsys-challenge-2015"
DIGINETICA_INPUT_DIR = (
    KAGGLE_INPUT_DIR / "datasets/profalbusdumbledore/diginetica-dataset"
)
YOOCHOOSE_SOURCE = YOOCHOOSE_INPUT_DIR / "yoochoose-clicks.dat"
DIGINETICA_SOURCE = DIGINETICA_INPUT_DIR / "train-item-views.csv"

RESULTS_DIR = OUTPUT_DIR / "results"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"

for directory in [RESULTS_DIR, CHECKPOINTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"output_dir={OUTPUT_DIR}")
print(f"Using Yoochoose source: {YOOCHOOSE_SOURCE}")
print(f"Using Diginetica source: {DIGINETICA_SOURCE}")

## Preprocess Sessions

The preprocessing flow builds ordered sessions, removes short sessions and rare items, splits chronologically, remaps item ids from training data, and expands sessions into prefix-label examples.

In [ ]:
def load_yoochoose_sessions(path):
    df = pd.read_csv(
        path,
        header=None,
        usecols=[0, 1, 2],
        names=["session_id", "timestamp", "item_id"],
    )
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.tz_convert(None)

    session_items = {}
    session_dates = {}
    current_session_id = None
    current_timestamp = None

    for row in df.itertuples(index=False):
        session_id = int(row.session_id)
        if current_timestamp is not None and current_session_id != session_id:
            session_dates[current_session_id] = current_timestamp

        current_session_id = session_id
        current_timestamp = row.timestamp

        if session_id in session_items:
            session_items[session_id].append(row.item_id)
        else:
            session_items[session_id] = [row.item_id]

    if current_session_id is not None:
        session_dates[current_session_id] = current_timestamp

    return [
        (session_id, session_dates[session_id], items)
        for session_id, items in session_items.items()
    ]


def load_diginetica_sessions(path):
    df = pd.read_csv(
        path,
        sep=";",
        usecols=["sessionId", "itemId", "timeframe", "eventdate"],
    )
    df = df.rename(columns={"sessionId": "session_id", "itemId": "item_id"})
    df["eventdate"] = pd.to_datetime(df["eventdate"], format="%Y-%m-%d")

    session_clicks = {}
    session_dates = {}
    current_session_id = None
    current_date = None

    for row in df.itertuples(index=False):
        session_id = int(row.session_id)
        if current_date is not None and current_session_id != session_id:
            session_dates[current_session_id] = current_date

        current_session_id = session_id
        current_date = row.eventdate

        click = (row.item_id, int(row.timeframe))
        if session_id in session_clicks:
            session_clicks[session_id].append(click)
        else:
            session_clicks[session_id] = [click]

    if current_session_id is not None:
        session_dates[current_session_id] = current_date

    sessions = []
    for session_id, clicks in session_clicks.items():
        ordered_clicks = sorted(clicks, key=lambda click: click[1])
        items = [item for item, _ in ordered_clicks]
        sessions.append((session_id, session_dates[session_id], items))
    return sessions

In [ ]:
def drop_short_sessions(sessions):
    return [session for session in sessions if len(session[2]) >= 2]


def drop_rare_items(sessions, min_freq=5):
    counts = Counter()
    for _, _, items in sessions:
        counts.update(items)

    result = []
    for session_id, date, items in sessions:
        kept = [i for i in items if counts[i] >= min_freq]
        if len(kept) >= 2:
            result.append((session_id, date, kept))
    return result


def sort_by_date(sessions):
    return sorted(sessions, key=lambda session: session[1])


def split_by_date(sessions, test_days):
    max_date = max(date for _, date, _ in sessions)
    split_date = max_date - pd.Timedelta(days=test_days)
    train = [s for s in sessions if s[1] < split_date]
    test = [s for s in sessions if s[1] > split_date]
    return train, test


def renumber_training_items(train_sessions):
    item_to_index = {}
    next_item_index = 1
    remapped_sessions = []

    for session_id, date, items in train_sessions:
        remapped_items = []
        for item in items:
            if item not in item_to_index:
                item_to_index[item] = next_item_index
                next_item_index += 1
            remapped_items.append(item_to_index[item])
        remapped_sessions.append((session_id, date, remapped_items))

    return remapped_sessions, item_to_index


def remap_test_sessions(test_sessions, item_to_index):
    remapped_sessions = []
    for session_id, date, items in test_sessions:
        remapped_items = [
            item_to_index[item] for item in items if item in item_to_index
        ]
        if len(remapped_items) >= 2:
            remapped_sessions.append((session_id, date, remapped_items))
    return remapped_sessions


def expand_sessions(sessions):
    examples = []
    for session_id, date, items in sessions:
        for reverse_offset in range(1, len(items)):
            examples.append(
                (session_id, date, items[:-reverse_offset], items[-reverse_offset])
            )
    return examples


def keep_recent_fraction(examples, denominator):
    if denominator is None:
        return examples
    keep = len(examples) // denominator
    return examples[-keep:] if keep else examples


def prefix_label_rows(examples):
    return [(list(prefix), int(label)) for _, _, prefix, label in examples]


def vocabulary_size_from_rows(*row_groups):
    max_item_id = 0
    for rows in row_groups:
        for prefix, label in rows:
            max_item_id = max(max_item_id, int(label), max(prefix))
    return max_item_id + 1


def preprocess(sessions, test_days, train_fraction_denominator):
    sessions = drop_short_sessions(sessions)
    sessions = drop_rare_items(sessions)
    sessions = sort_by_date(sessions)
    train_sessions, test_sessions = split_by_date(sessions, test_days)

    train_sessions, item_to_index = renumber_training_items(train_sessions)
    test_sessions = remap_test_sessions(test_sessions, item_to_index)

    train_examples = expand_sessions(train_sessions)
    test_examples = expand_sessions(test_sessions)
    train_examples = keep_recent_fraction(train_examples, train_fraction_denominator)
    return train_examples, test_examples

In [ ]:
yoochoose_sessions = load_yoochoose_sessions(YOOCHOOSE_SOURCE)
diginetica_sessions = load_diginetica_sessions(DIGINETICA_SOURCE)

yoochoose_1_64_train, yoochoose_1_64_test = preprocess(
    yoochoose_sessions, test_days=1, train_fraction_denominator=64
)
diginetica_train, diginetica_test = preprocess(
    diginetica_sessions, test_days=7, train_fraction_denominator=None
)

YOOCHOOSE_1_64_TRAIN_ROWS = prefix_label_rows(yoochoose_1_64_train)
YOOCHOOSE_1_64_TEST_ROWS = prefix_label_rows(yoochoose_1_64_test)
DIGINETICA_TRAIN_ROWS = prefix_label_rows(diginetica_train)
DIGINETICA_TEST_ROWS = prefix_label_rows(diginetica_test)

print(f"Yoochoose 1/64 train examples: {len(YOOCHOOSE_1_64_TRAIN_ROWS):,}")
print(f"Yoochoose 1/64 test examples: {len(YOOCHOOSE_1_64_TEST_ROWS):,}")
print(f"Diginetica train examples: {len(DIGINETICA_TRAIN_ROWS):,}")
print(f"Diginetica test examples: {len(DIGINETICA_TEST_ROWS):,}")

In [ ]:
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

if torch.cuda.is_available():
    print(f"cuda_device={torch.cuda.get_device_name(0)}")
    print(
        f"cuda_memory_gb={torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}"
    )
else:
    print("cuda_device=None")

print("Available Kaggle input files:")
for dirname, _, filenames in os.walk(KAGGLE_INPUT_DIR):
    for filename in filenames:
        print(Path(dirname) / filename)

## Data loading and graph construction helpers

In [ ]:
def build_session_graph(prefix, label):
    """Turn a `(prefix, label)` example into a PyG `Data` graph.

    Nodes are unique items in the prefix. Forward edges follow click order;
    backward edges expose reverse transition context, mirroring SR-GNN's
    separate incoming/outgoing adjacency channels.
    """
    unique_items = list(dict.fromkeys(prefix))
    item_to_node = {item: index for index, item in enumerate(unique_items)}
    click_sequence = [item_to_node[item] for item in prefix]

    edges = list(dict.fromkeys(zip(click_sequence[:-1], click_sequence[1:])))

    forward_sources, forward_targets = [], []
    backward_sources, backward_targets = [], []
    for source, target in edges:
        forward_sources.append(source)
        forward_targets.append(target)

        backward_sources.append(target)
        backward_targets.append(source)

    if forward_sources:
        forward_edge_index = torch.tensor(
            [forward_sources, forward_targets], dtype=torch.long
        )
        backward_edge_index = torch.tensor(
            [backward_sources, backward_targets], dtype=torch.long
        )
    else:
        forward_edge_index = torch.zeros((2, 0), dtype=torch.long)
        backward_edge_index = torch.zeros((2, 0), dtype=torch.long)

    return Data(
        x=torch.tensor(unique_items, dtype=torch.long),
        edge_index=forward_edge_index,
        forward_edge_index=forward_edge_index,
        backward_edge_index=backward_edge_index,
        sequence=torch.tensor(click_sequence, dtype=torch.long),
        sequence_length=torch.tensor(len(click_sequence), dtype=torch.long),
        last_click=torch.tensor(click_sequence[-1], dtype=torch.long),
        y=torch.tensor(label, dtype=torch.long),
        num_nodes=len(unique_items),
    )


class SessionGraphDataset(Dataset):
    """Lazy dataset of session-prefix graphs.

    Graphs are built on demand so training does not materialize all PyG
    objects in memory before training starts.
    """

    def __init__(self, prefix_label_rows):
        self.prefix_label_rows = prefix_label_rows

    def __len__(self):
        return len(self.prefix_label_rows)

    def __getitem__(self, index):
        prefix, label = self.prefix_label_rows[index]
        return build_session_graph(prefix, label)

## Load datasets and compute vocabulary size

In [ ]:
yoochoose_1_64_num_items = vocabulary_size_from_rows(
    YOOCHOOSE_1_64_TRAIN_ROWS, YOOCHOOSE_1_64_TEST_ROWS
)
diginetica_num_items = vocabulary_size_from_rows(
    DIGINETICA_TRAIN_ROWS, DIGINETICA_TEST_ROWS
)

print(f"Yoochoose 1/64 num_items: {yoochoose_1_64_num_items:,}")
print(f"Diginetica      num_items: {diginetica_num_items:,}")

## Bidirectional GAT Encoder

Multi-layer, multi-head GAT encoder shared by all three architectures.
Replaces the GGNN from SR-GNN / TAGNN.

In [ ]:
class DirectionalGATStack(nn.Module):
    """GAT stack for one transition direction."""

    def __init__(
        self,
        hidden_dim=100,
        num_layers=1,
        num_heads=4,
        dropout=0.1,
        concat_heads=True,
    ):
        super().__init__()

        if concat_heads and hidden_dim % num_heads != 0:
            raise ValueError(
                f"hidden_dim ({hidden_dim}) must be divisible by num_heads "
                f"({num_heads}) when concat_heads=True"
            )

        self.dropout = dropout
        per_head_dim = hidden_dim // num_heads if concat_heads else hidden_dim
        self.gat_layers = nn.ModuleList(
            GATConv(
                in_channels=hidden_dim,
                out_channels=per_head_dim,
                heads=num_heads,
                concat=concat_heads,
                dropout=dropout,
                add_self_loops=True,
            )
            for _ in range(num_layers)
        )

    def forward(self, node_features, edge_index):
        for gat_layer in self.gat_layers:
            node_features = F.elu(gat_layer(node_features, edge_index))
            node_features = F.dropout(
                node_features, p=self.dropout, training=self.training
            )
        return node_features


class BidirectionalGATEncoder(nn.Module):
    """Separate forward/backward GAT stacks with shared item embeddings."""

    def __init__(
        self,
        num_items,
        hidden_dim=100,
        num_layers=1,
        num_heads=4,
        dropout=0.1,
    ):
        super().__init__()
        self.embedding = nn.Embedding(num_items, hidden_dim, padding_idx=0)
        self.forward_gat = DirectionalGATStack(
            hidden_dim, num_layers, num_heads, dropout
        )
        self.backward_gat = DirectionalGATStack(
            hidden_dim, num_layers, num_heads, dropout
        )
        self.direction_fusion = nn.Linear(2 * hidden_dim, hidden_dim, bias=True)

    def forward(
        self,
        node_item_ids,
        forward_edge_index,
        backward_edge_index,
    ):
        node_features = self.embedding(node_item_ids)
        forward_features = self.forward_gat(
            node_features, forward_edge_index
        )
        backward_features = self.backward_gat(
            node_features, backward_edge_index
        )
        return self.direction_fusion(
            torch.cat([forward_features, backward_features], dim=-1)
        )

## GAT + SR-GNN-Style Readout


In [ ]:
class GATSRGNN(nn.Module):
    def __init__(
        self,
        num_items,
        hidden_dim=100,
        num_layers=1,
        num_heads=4,
        dropout=0.1,
    ):
        super().__init__()
        self.encoder = BidirectionalGATEncoder(
            num_items, hidden_dim, num_layers, num_heads, dropout
        )
        self.hidden_dim = hidden_dim

        self.sequence_attention_projection = nn.Linear(
            hidden_dim, hidden_dim, bias=False
        )
        self.last_click_attention_projection = nn.Linear(
            hidden_dim, hidden_dim, bias=False
        )
        self.attention_score_projection = nn.Linear(hidden_dim, 1, bias=False)
        self.hybrid_projection = nn.Linear(2 * hidden_dim, hidden_dim, bias=True)
        self.reset_parameters()

    def reset_parameters(self):
        with torch.no_grad():
            self.encoder.embedding.weight[0].fill_(0)

    def forward(self, batch):
        """Return logits [batch_size, num_items]."""
        node_hidden = self.encoder(
            batch.x,
            batch.forward_edge_index,
            batch.backward_edge_index,
        )

        sequence_hidden, sequence_batch = self._sequence_hidden(batch, node_hidden)
        sequence_lengths = batch.sequence_length.view(-1).long()
        sequence_offsets = torch.cat(
            [
                sequence_lengths.new_zeros(1),
                sequence_lengths.cumsum(dim=0)[:-1],
            ]
        )

        last_sequence_index = sequence_offsets + sequence_lengths - 1
        local_preference = sequence_hidden[last_sequence_index]

        expanded_local_preference = local_preference[sequence_batch]
        attention_logits = self.attention_score_projection(
            torch.sigmoid(
                self.sequence_attention_projection(sequence_hidden)
                + self.last_click_attention_projection(expanded_local_preference)
            )
        ).squeeze(-1)
        attention_weights = pyg_softmax(attention_logits, sequence_batch)
        attention_weights = attention_weights.to(sequence_hidden.dtype)

        global_preference = torch.zeros_like(local_preference)
        global_preference.scatter_add_(
            0,
            sequence_batch.unsqueeze(-1).expand_as(sequence_hidden),
            attention_weights.unsqueeze(-1) * sequence_hidden,
        )

        session_representation = self.hybrid_projection(
            torch.cat([local_preference, global_preference], dim=-1)
        )

        item_embeddings = self.encoder.embedding.weight
        logits = session_representation @ item_embeddings.T
        return logits

    @staticmethod
    def _sequence_hidden(batch, node_hidden):
        """Map contextual node embeddings back to original prefix positions."""
        sequence_lengths = batch.sequence_length.view(-1).long()
        sequence_parts = torch.split(batch.sequence, sequence_lengths.tolist())

        hidden_parts = []
        batch_parts = []
        for graph_index, local_sequence in enumerate(sequence_parts):
            node_offset = batch.ptr[graph_index]
            hidden_parts.append(node_hidden[node_offset + local_sequence])
            batch_parts.append(
                local_sequence.new_full((local_sequence.numel(),), graph_index)
            )

        return torch.cat(hidden_parts, dim=0), torch.cat(batch_parts, dim=0)

## Training and Evaluation Protocol

The SR-GNN paper evaluates with `P@20` and `MRR@20`. Its reported setup uses hidden size `100`, a random `10%` validation split from the training set, Adam with learning rate `0.001`, learning-rate decay by `0.1` every 3 epochs, batch size `100`, and L2 penalty `1e-5`.

This notebook uses that setup directly for the GAT + SR-GNN-style readout experiment.


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def random_train_validation_split(rows, validation_fraction=0.1, seed=42):
    indices = list(range(len(rows)))
    random.Random(seed).shuffle(indices)
    validation_size = max(1, int(len(indices) * validation_fraction))
    validation_indices = set(indices[:validation_size])

    train_rows = []
    validation_rows = []
    for index, row in enumerate(rows):
        if index in validation_indices:
            validation_rows.append(row)
        else:
            train_rows.append(row)
    return train_rows, validation_rows


def build_loader(rows, batch_size, shuffle, device):
    dataset = SessionGraphDataset(rows)
    use_cuda = device.type == "cuda"
    num_workers = 2 if use_cuda else 0
    return PyGDataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=use_cuda,
        persistent_workers=num_workers > 0,
    )


def mask_padding_item(logits):
    logits = logits.clone()
    logits[:, 0] = -torch.finfo(logits.dtype).max
    return logits

In [ ]:
def precision_mrr_at_k(logits, targets, k=20):
    k = min(k, logits.size(1))
    top_items = logits.topk(k, dim=1).indices
    matches = top_items.eq(targets.view(-1, 1))

    hits = matches.any(dim=1).float()
    ranks = torch.zeros(targets.size(0), device=logits.device)
    matched_rows, matched_cols = matches.nonzero(as_tuple=True)
    ranks[matched_rows] = matched_cols.float() + 1
    reciprocal_ranks = torch.where(ranks > 0, 1.0 / ranks, torch.zeros_like(ranks))

    return hits.sum().item(), reciprocal_ranks.sum().item(), targets.size(0)


@torch.no_grad()
def evaluate(model, loader, device, k=20):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    total_hits = 0.0
    total_mrr = 0.0

    for batch_index, batch in enumerate(loader):
        batch = batch.to(device, non_blocking=device.type == "cuda")
        logits = mask_padding_item(model(batch))
        loss = F.cross_entropy(logits, batch.y)
        assert_finite("evaluation logits", logits, batch_index)
        assert_finite("evaluation loss", loss, batch_index)

        hits, mrr, examples = precision_mrr_at_k(logits, batch.y, k=k)
        total_loss += loss.item() * examples
        total_examples += examples
        total_hits += hits
        total_mrr += mrr

    return {
        "loss": total_loss / total_examples,
        "precision@20": 100.0 * total_hits / total_examples,
        "mrr@20": 100.0 * total_mrr / total_examples,
        "examples": total_examples,
    }


def assert_finite(name, tensor, batch_index):
    if not torch.isfinite(tensor).all():
        raise RuntimeError(f"Non-finite {name} detected in batch {batch_index}")


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    total_examples = 0

    for batch_index, batch in enumerate(loader):
        batch = batch.to(device, non_blocking=device.type == "cuda")
        optimizer.zero_grad(set_to_none=True)
        logits = mask_padding_item(model(batch))
        loss = F.cross_entropy(logits, batch.y)
        assert_finite("logits", logits, batch_index)
        assert_finite("loss", loss, batch_index)

        loss.backward()
        optimizer.step()

        examples = batch.y.size(0)
        total_loss += loss.item() * examples
        total_examples += examples

    return total_loss / total_examples

In [ ]:
DATASETS = {
    "Yoochoose 1/64": {
        "train_rows": YOOCHOOSE_1_64_TRAIN_ROWS,
        "test_rows": YOOCHOOSE_1_64_TEST_ROWS,
        "num_items": yoochoose_1_64_num_items,
    },
    "Diginetica": {
        "train_rows": DIGINETICA_TRAIN_ROWS,
        "test_rows": DIGINETICA_TEST_ROWS,
        "num_items": diginetica_num_items,
    },
}

config = {
    "epochs": 30,
    "patience": 5,
    "batch_size": 100,
    "learning_rate": 0.001,
    "weight_decay": 1e-5,
    "lr_decay_step": 3,
    "lr_decay_gamma": 0.5,
    "validation_fraction": 0.1,
    "hidden_dim": 100,
    "num_layers": 1,
    "num_heads": 4,
    "dropout": 0.1,
    "seed": 42,
}

device = get_device()
print(f"device={device}")
print(config)

In [ ]:
def checkpoint_name(dataset_name):
    safe_name = dataset_name.lower().replace(" ", "_").replace("/", "_")
    return f"gat_sr_gnn_{safe_name}.pt"


def run_dataset_experiment(dataset_name, dataset, config, device):
    set_seed(config["seed"])
    train_rows = list(dataset["train_rows"])
    test_rows = list(dataset["test_rows"])
    train_rows, validation_rows = random_train_validation_split(
        train_rows,
        validation_fraction=config["validation_fraction"],
        seed=config["seed"],
    )

    train_loader = build_loader(
        train_rows, config["batch_size"], shuffle=True, device=device
    )
    validation_loader = build_loader(
        validation_rows, config["batch_size"], shuffle=False, device=device
    )
    test_loader = build_loader(
        test_rows, config["batch_size"], shuffle=False, device=device
    )

    model = GATSRGNN(
        num_items=dataset["num_items"],
        hidden_dim=config["hidden_dim"],
        num_layers=config["num_layers"],
        num_heads=config["num_heads"],
        dropout=config["dropout"],
    ).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=config["lr_decay_step"],
        gamma=config["lr_decay_gamma"],
    )
    history = []
    best_validation_mrr = -1.0
    best_validation_precision = -1.0
    best_epoch = 0
    best_state = None
    bad_counter = 0

    for epoch in range(1, config["epochs"] + 1):
        started_at = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        validation_metrics = evaluate(model, validation_loader, device)
        scheduler.step()

        row = {
            "dataset": dataset_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": validation_metrics["loss"],
            "validation_precision@20": validation_metrics["precision@20"],
            "validation_mrr@20": validation_metrics["mrr@20"],
            "epoch_seconds": time.time() - started_at,
            "train_examples": len(train_rows),
            "validation_examples": len(validation_rows),
            "test_examples": len(test_rows),
        }
        history.append(row)
        print(
            f"{dataset_name} epoch {epoch:02d} "
            f"loss={train_loss:.4f} "
            f"val_loss={row['validation_loss']:.4f} "
            f"val_P@20={row['validation_precision@20']:.2f} "
            f"val_MRR@20={row['validation_mrr@20']:.2f} "
            f"time={row['epoch_seconds']:.1f}s"
        )

        if validation_metrics["mrr@20"] >= best_validation_mrr:
            best_validation_mrr = validation_metrics["mrr@20"]
            best_validation_precision = validation_metrics["precision@20"]
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            bad_counter = 0
        else:
            bad_counter += 1
            if bad_counter >= config["patience"]:
                print(f"early stopping at epoch {epoch}; best epoch was {best_epoch}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    test_metrics = evaluate(model, test_loader, device)

    checkpoint_path = CHECKPOINTS_DIR / checkpoint_name(dataset_name)
    torch.save(
        {
            "dataset": dataset_name,
            "model": "GATSRGNN",
            "config": dict(config),
            "num_items": dataset["num_items"],
            "state_dict": model.state_dict(),
            "test_metrics": test_metrics,
            "history": history,
            "best_epoch": best_epoch,
            "best_validation_precision@20": best_validation_precision,
            "best_validation_mrr@20": best_validation_mrr,
        },
        checkpoint_path,
    )

    result = {
        "dataset": dataset_name,
        "test_precision@20": test_metrics["precision@20"],
        "test_mrr@20": test_metrics["mrr@20"],
        "test_loss": test_metrics["loss"],
        "best_epoch": best_epoch,
        "best_validation_precision@20": best_validation_precision,
        "best_validation_mrr@20": best_validation_mrr,
        "train_examples": len(train_rows),
        "validation_examples": len(validation_rows),
        "test_examples": len(test_rows),
        "num_items": dataset["num_items"],
        "checkpoint_path": str(checkpoint_path),
    }
    return result, history

In [ ]:
all_results = []
all_history = []

for dataset_name, dataset in DATASETS.items():
    result, history = run_dataset_experiment(dataset_name, dataset, config, device)
    all_results.append(result)
    all_history.extend(history)

results = pd.DataFrame(all_results)
history = pd.DataFrame(all_history)

results_path = RESULTS_DIR / "gat_sr_gnn_results.csv"
history_path = RESULTS_DIR / "gat_sr_gnn_history.csv"
results.to_csv(results_path, index=False)
history.to_csv(history_path, index=False)

display(results)
print(f"saved {results_path}")
print(f"saved {history_path}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

results.plot.bar(
    x="dataset",
    y="test_precision@20",
    ax=axes[0],
    legend=False,
    color="#3b6ea8",
    title="Test Precision@20",
)
axes[0].set_ylabel("%")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=25)

results.plot.bar(
    x="dataset",
    y="test_mrr@20",
    ax=axes[1],
    legend=False,
    color="#b45f3c",
    title="Test MRR@20",
)
axes[1].set_ylabel("%")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
figure_path = RESULTS_DIR / "gat_sr_gnn_metrics.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
print(f"saved {figure_path}")

In [ ]:
best_result = results.sort_values(
    ["best_validation_mrr@20", "best_validation_precision@20"],
    ascending=False,
).iloc[0]
best_dataset_slug = best_result["dataset"].lower().replace(" ", "-").replace("/", "-")
best_checkpoint_path = Path(best_result["checkpoint_path"])

MODEL_SLUG = "gat-sr-gnn-session-recommender"
VARIATION_SLUG = f"best-{best_dataset_slug}"

kagglehub.model_upload(
    handle=f"karolbystrek/{MODEL_SLUG}/pytorch/{VARIATION_SLUG}",
    local_model_dir=str(best_checkpoint_path.parent),
    version_notes=f"Best GAT-SR-GNN checkpoint selected by validation MRR@20. Update {date.today().isoformat()}",
)
print(f"uploaded {MODEL_SLUG}/{VARIATION_SLUG} from {best_checkpoint_path.parent}")